## More visualisations with Altair

- Last time we mentioned that the gallery of example visualisations https://altair-viz.github.io/gallery/index.html is a really helpful place to start with making a new visualisation. 

- A good way of getting started is to cut and paste an example that is close to looking like a visualisation you want to generate and adapt it from there. 

- If you haven't followed that link to look at some of the possible plots that can be made with Altair now is a good time to do that and just browse through to get a sense of what it can do and what the associated code looks like. 


## Reminder and expansion on the steps of Altair 

1. **Import Libraries**: Import necessary libraries such as Altair and Pandas to handle data and create visualizations.
   
2. **Prepare Your Data**: Load and prepare your data using Pandas, ensuring it's in a suitable format for visualization.
3. **Create a Base Chart**: Use `alt.Chart()` to define a base chart with your DataFrame as the starting point.
   
4. **Choose a Mark Type**: Select the type of visualization (e.g., bar, line, scatter) using the appropriate `mark_*` method.
   
5. **Encode Data**: Map data fields to visual properties like axes, colors, and sizes using the `encode()` method.
   
6. **Transform Data (Optional)**: Apply transformations such as filtering or aggregating to prepare data for visualization.
   
7. **Customize Chart Properties**: Adjust properties like width, height, and title to enhance the chart's appearance and readability.
   
8. **Combine Charts (Optional)**: Combine multiple charts using operators like `+`, `|`, or `&` for overlaying or arranging in grids.
   
9. **Display the Chart**: Render the chart in a Jupyter Notebook or save it to a file for sharing or further analysis.
   
Data type style convention in Altair

| Data Type     | Shorthand Code | Description                          |
|---------------|--------------|--------------------------------------|
| **quantitative** | 'Q'          | A continuous real-valued quantity   |
| **ordinal**      | 'O'          | A discrete ordered quantity         |
| **nominal**      | 'N'          | A discrete unordered category       |
| **temporal**     | 'T'          | A time or date value                |
| **geojson**      | 'G'          | A geographic shape                  |


### Energy production Multi-Ring visualisation

This website has a great live dashboard giving details about power generation in the UK https://grid.iamkate.com. 

This type of visualisation is a neat way of connecting the overall class of generation (fossil fuel/renewable/etc) with the exact breakdown in that class (gas/coal/solar/etc). We won't really deal with the dashboard aspect of this visualisation (it is really just some code behind the scenes that gets a feed of live data in a useable format, and then puts the visualisation on a live webpage), instead we've replicated a bit of the data in a static form in a spreadsheet. 

**The task therefore is to recreate the visualisation in altair in a slightly similar way.** 

First, let's import the modules we need, and load in our data and have a quick look at it. 

In [1]:
import pandas as pd
import altair as alt

energy = pd.read_excel('Energy_generation_over_last_year.xlsx')
print(energy.head())

                   Fuel  Generated value  Percentage       Energy type
0                  Coal             0.49         1.6       Fossil fuel
1  Gas (combined cycle)            12.06        39.9       Fossil fuel
2         Solar voltaic             1.22         4.0  Renewable energy
3                  Wind             5.79        19.1  Renewable energy
4         Hydroelectric             0.38         1.2  Renewable energy


### Pie-Chart

If we look at the sorts of visualisation on the webpage https://grid.iamkate.com they are done as round polar diagrams, partly a full pie chart style and then with an outer ring that is like a donut chart. 

Looking in the gallery we can find some details of how to produce a pie chart, https://altair-viz.github.io/gallery/pie_chart.html, so let's start there and just try plotting our data using the code from that page.

In [5]:
# Create an arc (pie) chart using the energy DataFrame.
alt.Chart(energy).mark_arc().encode(
    # Encode the 'Percentage' field as the theta (angle) of the arc, specifying it as quantitative data.
    theta=alt.Theta(field="Percentage", type="quantitative"),
    # Encode the 'Fuel' field as the color of the arc, specifying it as nominal data to differentiate categories.
    color=alt.Color(field="Fuel", type="nominal"),
)

alt.Chart(...)

- That's a pretty good start, it makes a pie chart that makes sense and summarises our data. But immediately there are some things I would fix `field="Percentage", type="quantitative"` is a bit long-winded, we could replace it with `'Percentage:Q'`. 

- Likewise for the colour we could write less and get the same result. I would also like to make the radius of the pie chart a bit smaller as it feels bigger than it need be, and bigger than the ones on the source website I am replicating. 

- Lastly the source website had visualisations that are interactive, when I hover over them I see a summary of some of the information. We can get similar interactivity on our visualisation in altair (we'll discuss in more detail in a later notebook). To get something to appear when we hover we use a `tooltip`, and specify what fields in our data should be shown in the box on hovering over with the mouse. So, let's go ahead and add something of that kind too.

Putting those three sets of changes together we get this:

In [6]:
# Create an arc (pie) chart using the energy DataFrame with a specified radius.
alt.Chart(energy).mark_arc(radius=100).encode(
    # Encode the 'Percentage' field as the theta (angle) of the arc, specifying it as quantitative data.
    theta=alt.Theta('Percentage:Q'),
    # Encode the 'Fuel' field as the color of the arc, specifying it as nominal data to differentiate categories.
    color=alt.Color('Fuel:N'),
    # Add a tooltip to display additional information when hovering over each arc segment.
    # The tooltip includes 'Fuel', 'Generated value', and 'Percentage' fields.
    tooltip=['Fuel', 'Generated value', 'Percentage']
)

alt.Chart(...)

So far so good, but we have focussed on the data that makes up the inner pie chart, next we need to think about the outer donut part of the visualisation. The data for that part is an aggregation (summarising if you prefer) of the data using the categories given in the column `'Energy type'`. 

Altair has various functions we can use called 'transforms' which allow you to plot summarised versions of your data in various ways (see https://altair-viz.github.io/user_guide/transform/index.html). Usefully for our purposes there is one called `transform_aggregate()` that allows us to do precisely the thing we want to do and aggregate our different generation methods by adding them together depending on which `'Energy type'` they are.

Here's how we go about that for a pie chart. 

- We make our pie chart using a new field which we can make up a new name for `'Overall Percentage'`. 
- We then use our new aggregate function and need to define that new field as `sum(Percentage)` to sum up all the values in the `Percentage` field but aggregated by `'Energy type'` as noted by the `groupby=['Energy type']`.

Here's that working together:

In [7]:
# Create an arc (pie) chart using the energy DataFrame with a specified radius.
alt.Chart(energy).mark_arc(radius=100).encode(
    # Encode the 'Overall Percentage' field as the theta (angle) of the arc, specifying it as quantitative data.
    theta=alt.Theta('Overall_Percentage:Q'),
    # Encode the 'Energy' field as the color of the arc, specifying it as nominal data to differentiate categories.
    color=alt.Color('Energy type:N')
).transform_aggregate(
    # created by summing up the values in the Percentage field.
    Overall_Percentage='sum(Percentage)',
    # groups the data by Energy type and then calculates the sum of Percentage values within each group
    groupby=['Energy type']
)

alt.Chart(...)

Let's go a step further and add an interaction. 

- Suppose I want to add the amount of energy generated in each `'Energy type'` as well (just as in the original I'm replicating), to do that I will need to define another new field in my aggregation. 
- So let's do that, we can make a new field called `Overall_generated_value` and create it as `'sum(Generated value)` to get the amount of energy added up with that `'Energy type'`.
- Lastly we add the new fields we calculated into the `tooltip` so that they appear when we hover the mouse over the plot.

In [8]:
# Create an arc (pie) chart using the energy DataFrame with a specified radius.
alt.Chart(energy).mark_arc(radius=100).encode(
    # Encode the 'Overall_Percentage' field as the theta (angle) of the arc, specifying it as quantitative data.
    theta=alt.Theta('Overall_Percentage:Q'),
    # Encode the 'Energy type' field as the color of the arc, specifying it as nominal data to differentiate categories.
    color=alt.Color('Energy type:N'),
    # Add a tooltip to display additional information when hovering over each arc segment.
    # The tooltip includes 'Energy type', 'Overall_generated_value', and 'Overall_Percentage' fields.
    tooltip=['Energy type:N', 'Overall_generated_value:Q', 'Overall_Percentage:Q']
).transform_aggregate(
    # Aggregate the data to calculate the total percentage for each energy type.
    Overall_Percentage='sum(Percentage)',
    # Aggregate the data to calculate the total generated value for each energy type.
    Overall_generated_value='sum(Generated value)',
    # Group the aggregation by 'Energy type' to perform the calculations for each unique energy type.
    groupby=['Energy type']
)


alt.Chart(...)

### Donut graphs

This is looking great for the sort of data and almost the right type of visualisation for the outer ring, but it is a full pie chart not a donut. 

So let's go back now to the gallery and look at the donut chart example in the gallery and try and replicate that with our data. 
- The page for a donut chart is here https://altair-viz.github.io/gallery/donut_chart.html, so using our modifications from before (simplifying the field specifications, making the overall radius a bit smaller, and adding our interaction), we get:

In [9]:
# Create a donut chart using the energy DataFrame with specified inner and outer radii.
alt.Chart(energy).mark_arc(innerRadius=50, radius=100).encode(
    # Encode the 'Percentage' field as the theta (angle) of the arc, specifying it as quantitative data.
    theta=alt.Theta('Percentage:Q'),
    # Encode the 'Fuel' field as the color of the arc, specifying it as nominal data to differentiate categories.
    color=alt.Color('Fuel:N'),
    # Add a tooltip to display additional information when hovering over each arc segment.
    # The tooltip includes 'Fuel', 'Generated value', and 'Percentage' fields.
    tooltip=['Fuel', 'Generated value', 'Percentage']
)

alt.Chart(...)

### Combining Pie-Chart and Donut graphs

This looks good for a donut form of our main data (and it looks like the equivalent of our original pie chart).

- Next we want to put the two parts we have been working on together. 
- We have seen already we can do that by calculating both charts separately and then 'adding' them together to get them to appear on top of each other. 
- The thing we need to do is just make sure that the radius of each part makes sense and fits together. But, doing that and putting together what we have to build this overall visualisation up, gives the following:

In [10]:
# Create a pie chart using the energy DataFrame with a specified radius.
chart1 = alt.Chart(energy).mark_arc(radius=50).encode(
    # Encode the 'Overall_Percentage' field as the theta (angle) of the arc, specifying it as quantitative data.
    theta=alt.Theta('Overall_Percentage:Q'),
    # Encode the 'Energy type' field as the color of the arc, specifying it as nominal data to differentiate categories.
    color=alt.Color('Energy type:N'),
    # Add a tooltip to display additional information when hovering over each arc segment.
    # The tooltip includes 'Energy type', 'Overall_generated_value', and 'Overall_Percentage' fields.
    tooltip=['Energy type:N', 'Overall_generated_value:Q', 'Overall_Percentage:Q']
).transform_aggregate(
    # Aggregate the data to calculate the total percentage for each energy type.
    Overall_Percentage='sum(Percentage)',
    # Aggregate the data to calculate the total generated value for each energy type.
    Overall_generated_value='sum(Generated value)',
    # Group the aggregation by 'Energy type' to perform the calculations for each unique energy type.
    groupby=['Energy type']
)

# Create a donut chart using the energy DataFrame with specified inner and outer radii.
chart2 = alt.Chart(energy).mark_arc(innerRadius=50, radius=100).encode(
    # Encode the 'Percentage' field as the theta (angle) of the arc, specifying it as quantitative data.
    theta=alt.Theta('Percentage:Q'),
    # Encode the 'Fuel' field as the color of the arc, specifying it as nominal data to differentiate categories.
    color=alt.Color('Fuel:N'),
    # Add a tooltip to display additional information when hovering over each arc segment.
    # The tooltip includes 'Fuel', 'Generated value', and 'Percentage' fields.
    tooltip=['Fuel', 'Generated value', 'Percentage']
)

# Combine chart1 and chart2, overlaying the pie chart and donut chart.
chart1 + chart2

alt.LayerChart(...)

**Question:** Oh no! This is almost there, but we can see something unexpected has happened - can you spot the issue? 

So, altair takes fields we mark as 'nominal' as containing words, basically as categories within that field of our data. For us `'Energy types'` and `'Fuel'` are both examples of this type of data. 

- What altair does by default when it gets a field of that type is just sort them alphabetically, which when the ordering is not important (which is usually the case for nominal data) is a reasonable choice. 
- Sadly here for us this has led to a problem, the inner and outer part of our chart don't line up as we would want them to (for example gas falls partly aligned with fossil fuel, but partly with other as well). 
- This is just simply a result of both parts of the chart being arranged alphabetically.

How do we fix this? Well, for us the order of our nominal field is important. and so we need to specify the ordering. Here again is another point where we would look in the documentation to get some inspiration for how to go about this. 
- The part of the documentation we might use to get into detail such as this is the API documentation here https://altair-viz.github.io/user_guide/api.html. This really details all of the different functions and options within them that we can use. If we look under `Encoding Channels` in the API documentation we can find a function that deals with the ordering of a field we are using in our chart https://altair-viz.github.io/user_guide/generated/channels/altair.Order.html. 

- That documentation shows that we can specify for our field to be ordered by looking at a different field in the data. So that gives us a way out of our problem, we could make the outer donut have an extra order statement to specify that the plot should be ordered by the `'Energy type'` used in the inner plot. So, here goes, let's try that and see:

In [11]:
chart1 = alt.Chart(energy).mark_arc(radius=50).encode(
    # Encode the 'Overall_Percentage' field as the theta (angle) of the arc, specifying it as quantitative data.
    theta=alt.Theta('Overall_Percentage:Q'),
    # Encode the 'Energy type' field as the color of the arc, specifying it as nominal data to differentiate categories.
    color=alt.Color('Energy type:N'),
    # The tooltip includes 'Energy type', 'Overall_generated_value as Q', and 'Overall_Percentage as Q' fields.
    tooltip=['Energy type:N','Overall_generated_value:Q','Overall_Percentage:Q']
).transform_aggregate(
    # Aggregate the data to calculate the total percentage for each energy type.
    Overall_Percentage='sum(Percentage)',
    # Aggregate the data to calculate the total generated value for each energy type.
    Overall_generated_value='sum(Generated value)',
    # Group the aggregation by 'Energy type' to perform the calculations
    groupby=['Energy type']
)

chart2 = alt.Chart(energy).mark_arc(innerRadius=50,radius=100).encode(
    # Encode the 'Percentage' field as the theta (angle) of the arc, specifying it as Q type.
    theta=alt.Theta('Percentage:Q'),
    # Encode the 'Fuel' field as the color of the arc, specifying it as nominal data to differentiate categories.
    color=alt.Color('Fuel:N'),
    # The specific order is done based one 'Energy type' variable
    order=alt.Order('Energy type:N'),
    # The tooltip includes 'Fuel', 'Generated value as Q', and 'Percentage as Q' fields.
    tooltip=['Fuel:N','Generated value:Q','Percentage:Q']
)

# Combining these two plots with plus sign on the same figure area
chart1 + chart2

alt.LayerChart(...)

### Changing the colours

Finally that has sorted out our arrangement and ordering. We are very nearly there. So what else is left? 

I don't like the colour scheme here. So let's change that. We can do that by taking the color encoding and adding something altair calls a scale as part of the `color` part of our chart https://altair-viz.github.io/user_guide/generated/channels/altair.Color.html. Altair knows about a lot of colour schemes and they are listed here https://vega.github.io/vega/docs/schemes/#reference, I'll choose one with purples and oranges for no reason but that it is nice-looking! We can do that by adding `scale=alt.Scale(scheme='purpleorange')` to the `color` statement in the first of our defined charts.

Lastly the visualisation we are replicating has no legend with field names and colors listed, they just rely on the mouse hover interaction instead. For this split type of visualisation this might be a bit easier, and I would like to copy that feature, so for both `color` definitions with my two charts I will set `legend=None` which just turns off the legend being produced.

So at the end here is what we get:

In [12]:
chart1 = alt.Chart(energy).mark_arc(radius=100).encode(
    # Encode the 'Overall_Percentage' field as the theta (angle) of the arc, specifying it as quantitative data.
    theta=alt.Theta('Overall_Percentage:Q'),
    # Encode the 'Energy type' field as the color of the arc, specifying  based on a specific scheme now 
    color=alt.Color('Energy type:N',legend = None, scale=alt.Scale(scheme='purpleorange')),
    # The tooltip includes 'Energy type', 'Overall_generated_value as Q', and 'Overall_Percentage as Q' fields.
    tooltip=['Energy type','Overall_generated_value:Q','Overall_Percentage:Q']
).transform_aggregate(
    # Aggregate the data to calculate the total percentage for each energy type.
    Overall_Percentage='sum(Percentage)',
    # Aggregate the data to calculate the total generated value for each energy type.
    Overall_generated_value='sum(Generated value)',
    # Group the aggregation by 'Energy type' to perform the calculations
    groupby=['Energy type']
)

chart2 = alt.Chart(energy).mark_arc(innerRadius=50,radius=100).encode(
    # Encode the 'Percentage' field as the theta (angle) of the arc, specifying it as Q type.
    theta=alt.Theta('Percentage:Q'),
    # Encode the 'Fuel' field as the color of the arc, specifying it as nominal data and no legend now.
    color=alt.Color('Fuel:N',legend = None),
    # The specific order is done based one 'Energy type' variable
    order=alt.Order('Energy type'),
    # The tooltip includes 'Fuel', 'Generated value as Q', and 'Percentage as Q' fields.
    tooltip=['Fuel:N','Generated value:Q','Percentage:Q']
)

chart1 + chart2

alt.LayerChart(...)

Even if it isn't identical to the one in the original webpage, but it is close enough to work out on step-by-step creation of one of the images given in the website. We've also seen some things about ordering fields, making interactions, and aggregation of our data in altair. 

**Remark:** *Be aware that there would have been other ways to produce the same plot (indeed there will usually be more than one way to produce something - none of which typically is objectively 'right').* 

- If you were thinking about using pandas to do the aggregation, or excel, and then producing the plot using two sources of data, indeed that would have been quite possible. 
- It is possible to label the fields with `'a) Fossil fuels'` and so on to get the ordering to work properly by making things work alphabetically. 
- Many reasonable solutions exist to the problems we thought about and worked through in developing this visualisation.

## Open-ended Exercises

Any of the results or explorations you make of these exercises could be added to your portfolio that you need to submit for assessment. Feel free to explore different alternatives by benefiting from the Altair examples and guidance. You can also try the similar tasks in Tableau by using the given data sets, if you like.

By using the same data set above, `Energy_generation_over_last_year.xlsx` try to explore the following alternatives

- a) Going back to the simple case in the first stage, and create another version as pie chart with labels, similar to the example given here: https://altair-viz.github.io/gallery/pie_chart_with_labels.html. You can imagine to sue labels from `Fuel` or `Energy Type` categories. 

- b) To extend the labeled pie chart with a different style, consider the labeling based on `Energy Type` for simplicity and create a radial chart that represents the values over different categories, by inspring from the example given here: https://altair-viz.github.io/gallery/radial_chart.html

- c) What if we combine new charts on a same figure or create a side-by-side outcome from a) and b)